# setup

In [1]:
import os
import sys
sys.path.insert(0,'/root/eallmancy')
sys.path.insert(0,'/root/eallmancy/experiments')
os.chdir('/root/eallmancy')

# TODO: THIS IS ULTRA HACKY, PUT API SETTINGS IN CONTROL PANEL
with open('/root/openai_api_key') as f:
    os.environ['OPENAI_API_KEY'] = f.read().replace('\n','')
with open('/root/gemini_api_key') as f:
    os.environ['GEMINI_API_KEY'] = f.read().replace('\n','')

from eallmancy.engine.llm_solution_engine_v3 import EngineV3
from eallmancy import Artifact
from eallmancy import SinglePartSolution, ProblemSpec
from eallmancy.engine.engine_trace import EngineTrace
from collections import defaultdict
import datetime
import numpy as np
import os
import re




In [2]:

from problems import TSPACOProblemSpec

CHATGPT_4OMINI_API = dict(
            host="api.openai.com",
            key=os.environ.get('OPENAI_API_KEY'),
            model="gpt-4o-mini",
            timeout=20
        )

GEMINI_25FLASH_API = dict(
            base_url='https://generativelanguage.googleapis.com/v1beta',
            api_key=os.environ.get('GEMINI_API_KEY'),
            model='gemini-2.0-flash',
            timeout=20,
            openai_format=True
        )

# data loading

In [3]:
from eallmancy.analysis.trace_analysis import TraceAnalysis
from copy import copy, deepcopy

def apply_analysis_proto(original_trace, trace_analysis_obj=None, trace_analysis_class=TraceAnalysis, **kwargs):
    
    new_trace = EngineTrace(problem_spec=trace_res.problem_spec)
    if trace_analysis_obj is not None:
        new_trace.trace_analysis = trace_analysis_obj
    else:
        new_trace.trace_analysis = (
            trace_analysis_class(
                ps=original_trace.problem_spec,
                idea_extractor_llm=CHATGPT_4OMINI_API,
                **kwargs
            )
        )
    
    for art in original_trace.list:
        new_art = copy(art)
        new_art.trace_id = None
        parent_arts = [trace_res[pn] for pn, _ in original_trace.graph.in_edges(art.trace_id)]
        new_trace.register_artifact(new_art, parent_arts)
    
    return new_trace


def _iterate_analysis(new_art, trace, last_trace_analysis):
    new_trace = EngineTrace(problem_spec=trace_res.problem_spec)
    new_trace.trace_analysis = deepcopy(last_trace_analysis)
    new_art = copy(new_art)
    new_art.trace_id = None
    parent_arts = [trace_res[pn] for pn, _ in original_trace.graph.in_edges(art.trace_id)]
    new_trace.register_artifact(new_art, parent_arts)
    return new_trace.trace_analysis
    
    

def simulate_analysis(original_trace, current_trace, start_art=None, end_art=None):
    
    new_trace = copy(current_trace)
    new_trace.trace_analysis = deepcopy(current_trace.trace_analysis)
    

    for art in original_trace.list:
        if start_art is not None and art.trace_id < start_art:
            continue
        if end_art is not None and art.trace_id >= end_art:
            break
        new_art = copy(art)
        new_art.trace_id = None
        parent_arts = [original_trace[pn] for pn, _ in original_trace.graph.in_edges(art.trace_id)]
        new_trace.register_artifact(new_art, parent_arts)
    
    return new_trace


In [4]:

def simulate_analysis_history(original_trace, stepsize=20, max_trace_id=None):
    
    trace_cache = dict()
    
    curr_trace = EngineTrace(problem_spec=original_trace.problem_spec, trace_analysis=True, idea_extractor_llm=CHATGPT_4OMINI_API,)
    
    last_idx = 0
    for idx in range(stepsize, min(max_trace_id, len(original_trace.list)), stepsize):
        curr_trace = trace_cache[idx] = simulate_analysis(original_trace, curr_trace, start_art=last_idx, end_art=idx)
        last_idx = idx
    else:
        if last_idx < len(original_trace.list):
            curr_trace = trace_cache[max_trace_id or len(original_trace.list)] = simulate_analysis(original_trace, curr_trace, start_art=last_idx, end_art=max_trace_id)
    
    return trace_cache

## testing insane overwrite

In [5]:

import polars as pl
import plotly.express as px

def trace_dist_matrixes(trace, **dist_funcs):
    "dist_func needs to take two artifacts and figure out the rest"
    
    idx_to_soln = {idx:_a for idx,_a in enumerate([a for a in trace.list if a.is_solution])}
    
    _plotrows = []
    for a in range(len(idx_to_soln)):
        for b in range(len(idx_to_soln)):
            sa = idx_to_soln[a]
            sb = idx_to_soln[b]
            e = dict(
                a_idx=a,
                b_idx=b,
                a_tidx=idx_to_soln[a].trace_id,
                b_tidx=idx_to_soln[b].trace_id,
                dist=0,
                a_fit=sa.fitness,
                b_fit=sb.fitness
            )
            if a==b:
                for k,_ in dist_funcs.items():
                    e[f'dist_{k}'] = 0
            else:
                for k,dist_func in dist_funcs.items():
                    e[f'dist_{k}'] = dist_func(sa, sb)
            _plotrows.append(e)
    _plotdf = pl.DataFrame(_plotrows)
    return _plotdf

def trace_dist_matrix(trace, dist_func):
    _plotdf = trace_dist_matrixes(trace, func=dist_func)
    return _plotdf.with_columns(dist=pl.col.dist_func).drop('dist_func')

def plot_trace_dist_matrix(trace, dist_func, _plotdf=None):
    if _plotdf is None:
        _plotdf = trace_dist_matrix(trace, dist_func)
    
    fig = px.scatter(
        _plotdf,
        x='a_idx',
        y='b_idx',
        color='dist',
        height=700,width=700
    )
    fig.update_layout(coloraxis_reversescale=True)
    fig.update_traces(marker_symbol="square")
    return fig

# trace_dist_matrix(original_trace, ta.distance)

## diversity, fitness 'modeling'

In [6]:
def shannon_wiener_clusters(arts, dist_func, thresh=0.1, max_trace_id=None):
    clusters = [] # list of list of artifacts

    for a in arts:
        if not a.is_solution:
            continue

        if max_trace_id is not None:
            if a.trace_id > max_trace_id:
                continue

        for c_idx, cl in enumerate(clusters):
            if any([dist_func(a, mem) > thresh for mem in cl]):
                continue  # someone broke similarity threshold
            clusters[c_idx].append(a)
            break
        else:
            clusters.append([a])  # new cluster
    return clusters

def shannon_wiener_probs(arts, *args, **kwargs):
    clusters = shannon_wiener_clusters(arts, *args, **kwargs)
    num_solns = sum([len(c) for c in clusters])
    return [len(c)*1.0/num_solns for c in clusters]

def shannon_wiener_diversity(arts, *args, **kwargs):
    probs = shannon_wiener_probs(arts, *args, **kwargs)
    return sum([-1*p*np.log2(p) for p in probs])

In [7]:

import networkx as nx


def cdi(arts, dist_func, max_trace_id=None):
    # clone trace graph up until max_trace_id, throwing away non-solutions

    solns = [a for a in arts if a.is_solution]
    if max_trace_id is not None:
        solns = [a for a in solns if a.trace_id <= max_trace_id]
    tmpG = nx.Graph()
    for a in solns:
        for b in solns:
            if a.trace_id >= b.trace_id:
                continue
            tmpG.add_edge(a.trace_id, b.trace_id, weight=dist_func(a,b))

    # construct minimium spanning tree
    mst_edge_weights = [d['weight'] for _, _, d in nx.minimum_spanning_edges(tmpG, data=True, weight="weight")]

    # probs as edge % of total weights
    total_mst_weight = sum(mst_edge_weights)
    mst_edge_probs = [ew*1.0/total_mst_weight for ew in mst_edge_weights]

    # calc entropy
    return sum([-1*p*np.log2(p) for p in mst_edge_probs])


In [8]:

import polars as pl
import plotly.express as px
from eallmancy.analysis.analyzer.analyzer import Analyzer


def top_k_generator(idx_to_soln_map, k=10):
    topk_cache = []
    for i in range(max(idx_to_soln_map.keys())):
        topk_cache.append(idx_to_soln_map[i])
        if len(topk_cache) > k:
            topk_cache = list(sorted(topk_cache, key=lambda a: a.fitness, reverse=True))[:k]
        yield i, topk_cache


def _plotdf_v9(trace):
    
    ta = Analyzer.build_analysis(trace, trace.problem_spec)
    idx_to_soln = {idx:_a for idx,_a in enumerate([a for a in trace.list if a.is_solution])}
    
    _plot_rows=[]
    _all_so_far = []
    for i, topk_cache in top_k_generator(idx_to_soln, k=10):
        _all_so_far.append(idx_to_soln[i])
        ee = dict(
            idx=i,
            g_cdi=cdi(topk_cache, ta.distance),
            h_cdi=cdi(_all_so_far, ta.distance),
            g_swdi=shannon_wiener_diversity(topk_cache, ta.distance),
            h_swdi=shannon_wiener_diversity(_all_so_far, ta.distance),
            best_so_far=max([a.fitness for a in _all_so_far]),
            fitness=idx_to_soln[i].fitness,
        )
        ee['is_breakthrough'] = ee['fitness'] == ee['best_so_far']
        _plot_rows.append(ee)
    _plotdf = pl.DataFrame(_plot_rows)
    return _plotdf


In [9]:
from eallmancy.analysis.analyzer.utils import get_codet5p_embeddings, get_openai_embeddings, cosine_sim

class PureEmbeddingAnalysis:
    def __init__(self, trace, embedding_func):
        all_solns = [a.trace_id for a in trace.list if a.is_solution]
        embs = embedding_func([CodeT5PAnalysis.get_clean_program(trace[ti]) for ti in all_solns])
        self.embs = {
            ti: emb for ti, emb in zip(all_solns, embs)
        }

    @classmethod
    def get_clean_program(cls, arte):
        return cls._clean_program(arte.program)
        
    @classmethod
    def _clean_program(cls, prog):
        # find function line
        fsig_idx = prog.find('def ')
        programbody_idx = fsig_idx + prog[fsig_idx:].find('\n')+1
        return prog[programbody_idx:]
        
    def distance(self, a, b):
        return 1 - cosine_sim(self.embs[a.trace_id], self.embs[b.trace_id])
        
class CodeT5PAnalysis(PureEmbeddingAnalysis):
    def __init__(self, trace):
        super().__init__(trace, embedding_func=get_codet5p_embeddings)
        
class OpenAIAnalysis(PureEmbeddingAnalysis):
    def __init__(self, trace):
        super().__init__(trace, embedding_func=get_openai_embeddings)

## distance vs fitness stuff

In [10]:
%%time
import glob
import pickle as pk

_plotdfs = []
for tid, tracepk in enumerate(glob.glob('/root/elmcy_reevo_traces/*.pk')):
    # if not 'op_aco' in tracepk:
    #     continue
    print('loading',tracepk)
    problem = tracepk[tracepk.rfind('reevo_')+6:tracepk.rfind('.pk')]
    trace = EngineTrace.load(tracepk)
    tb = CodeT5PAnalysis(trace)
    tc = OpenAIAnalysis(trace)
    # ta = Analyzer.build_analysis(trace, trace.problem_spec)
    with open(tracepk.replace('elmcy_reevo_traces', 'elmcy_reevo_analyses'), 'rb') as f:
        ta = pk.load(f)
    dd = trace_dist_matrixes(
            trace,
            elmcy=ta.distance,
            jplag=lambda a,b : ta.distance(a, b, use_cosim=False),
            codet5p=tb.distance,
            openai=tc.distance,
        
        ).with_columns(problem=pl.lit(problem))
    del ta
    del tb
    del tc
    # dd = trace_dist_matrix(trace, ta.distance)
    _plotdfs.append(dd)
    print('    loaded!')
_plotdf = pl.concat([ppd.with_columns(sim_idx=pl.lit(did)) for did, ppd in enumerate(_plotdfs)])


loading /root/elmcy_reevo_traces/20250802-235310_a266_reevo_op_aco.pk
    loaded!
loading /root/elmcy_reevo_traces/20250803-010854_7f89_reevo_op_aco.pk
    loaded!
loading /root/elmcy_reevo_traces/20250803-013958_8849_reevo_tsp_aco.pk
    loaded!
loading /root/elmcy_reevo_traces/20250802-235352_c6cb_reevo_op_aco.pk
    loaded!
loading /root/elmcy_reevo_traces/20250803-002048_663d_reevo_tsp_aco.pk
    loaded!
loading /root/elmcy_reevo_traces/20250803-010407_f23e_reevo_tsp_aco.pk
    loaded!
loading /root/elmcy_reevo_traces/20250803-015601_2e2b_reevo_cvrp_aco.pk
    loaded!
loading /root/elmcy_reevo_traces/20250803-020128_2611_reevo_cvrp_aco.pk
    loaded!
loading /root/elmcy_reevo_traces/20250802-235331_c803_reevo_op_aco.pk
    loaded!
loading /root/elmcy_reevo_traces/20250803-002027_2a2c_reevo_tsp_aco.pk
    loaded!
loading /root/elmcy_reevo_traces/20250803-011402_e009_reevo_op_aco.pk
    loaded!
loading /root/elmcy_reevo_traces/20250803-013004_c0d9_reevo_op_aco.pk
    loaded!
loading 

In [11]:
print(2)

2


In [13]:
#_plotdf.write_csv('_plotdf.csv')

In [ ]:

def dist_regularised(dist_col):
    return pl.col(dist_col) / pl.col(dist_col).max().over(['problem'])

fig = px.box(
    (
        _plotdf
        .filter(pl.col.a_fit != pl.lit(-float('inf')))
        .filter(pl.col.b_fit != pl.lit(-float('inf')))
        .with_columns(delta_fit=pl.col.a_fit - pl.col.b_fit)
        .with_columns(dist_bin=(dist_regularised('dist_jplag')/0.1).round()*0.1)
    ),
    x='dist_bin',
    y='delta_fit',
    facet_row='problem',
    color='problem',
    height=1000,
    # box=True,
    points=False,
    # color='sim_idx'
)
# fig.update_traces(showwhiskers=False)  # no whiskers (min/max), just Q1–Q3 + median
fig.update_yaxes(matches=None)

In [ ]:

def dist_regularised(dist_col):
    return pl.col(dist_col) / pl.col(dist_col).max().over(['problem'])

fig = px.box(
    (
        _plotdf
        .filter(pl.col.a_fit != pl.lit(-float('inf')))
        .filter(pl.col.b_fit != pl.lit(-float('inf')))
        .with_columns(delta_fit=pl.col.a_fit - pl.col.b_fit)
        # .with_columns(dist_bin=(dist_regularised('dist_openai')/0.1).round()*0.1)
        .with_columns(dist_bin=(pl.col('dist_openai')/0.1).round()*0.1)
    ),
    x='dist_bin',
    y='delta_fit',
    facet_row='problem',
    color='problem',
    height=1000,
    # box=True,
    points=False,
    # color='sim_idx'
)
# fig.update_traces(showwhiskers=False)  # no whiskers (min/max), just Q1–Q3 + median
fig.update_yaxes(matches=None)

In [ ]:

def dist_regularised(dist_col):
    return pl.col(dist_col) / pl.col(dist_col).max().over(['problem'])

fig = px.box(
    (
        _plotdf
        .filter(pl.col.a_fit != pl.lit(-float('inf')))
        .filter(pl.col.b_fit != pl.lit(-float('inf')))
        .with_columns(delta_fit=pl.col.a_fit - pl.col.b_fit)
        .with_columns(dist_bin=(dist_regularised('dist_codet5p')/0.1).round()*0.1)
    ),
    x='dist_bin',
    y='delta_fit',
    facet_row='problem',
    color='problem',
    height=1000,
    # box=True,
    points=False
    # color='sim_idx'
)
# fig.update_traces(showwhiskers=False)  # no whiskers (min/max), just Q1–Q3 + median
fig.update_yaxes(matches=None)


In [ ]:

def dist_regularised(dist_col):
    return pl.col(dist_col) / pl.col(dist_col).max().over(['problem'])

fig = px.box(
    (
        _plotdf
        .filter(pl.col.a_fit != pl.lit(-float('inf')))
        .filter(pl.col.b_fit != pl.lit(-float('inf')))
        .with_columns(delta_fit=pl.col.a_fit - pl.col.b_fit)
        .with_columns(dist_bin=(dist_regularised('dist_elmcy')/0.1).round()*0.1)
    ),
    x='dist_bin',
    y='delta_fit',
    facet_row='problem',
    color='problem',
    height=1000,
    # box=True,
    points=False
    # color='sim_idx'
)
# fig.update_traces(showwhiskers=False)  # no whiskers (min/max), just Q1–Q3 + median
fig.update_yaxes(matches=None)


## rolling top-K diversity, historical diversity, fitness improvement modelling

In [98]:
trace = original_trace
idx_to_soln = {idx:_a for idx,_a in enumerate([a for a in trace.list if a.is_solution])}

import polars as pl
import plotly.express as px


def top_k_generator(idx_to_soln_map, k=10):
    topk_cache = []
    for i in range(max(idx_to_soln.keys())):
        topk_cache.append(idx_to_soln[i])
        if len(topk_cache) > k:
            topk_cache = list(sorted(topk_cache, key=lambda a: a.fitness, reverse=True))[:k]
        yield i, topk_cache

_plot_rows=[]
_all_so_far = []
for i, topk_cache in top_k_generator(idx_to_soln, k=10):
    _all_so_far.append(idx_to_soln[i])
    ee = dict(
        idx=i,
        g_cdi=cdi(topk_cache, ta.distance),
        h_cdi=cdi(_all_so_far, ta.distance),
        g_swdi=shannon_wiener_diversity(topk_cache, ta.distance),
        h_swdi=shannon_wiener_diversity(_all_so_far, ta.distance),
        best_so_far=max([a.fitness for a in _all_so_far]),
        fitness=idx_to_soln[i].fitness,
    )
    ee['is_breakthrough'] = ee['fitness'] == ee['best_so_far']
    _plot_rows.append(ee)
_plot_df = pl.DataFrame(_plot_rows)

In [ ]:
fig = px.line(
    (
        _plotdf.filter(pl.col.idx>5).sort('h_swdi')
    ),
    x='h_swdi',
    y='best_so_far',
    line_group='sim_idx'
)
fig

In [ ]:
fig = px.line(
    (
        _plotdf.unpivot(index=['sim_idx', 'idx'])
        .with_columns(
            fitwin=pl.when(pl.col.variable.is_in(['best_so_far']))
                        .then(pl.lit('bfit'))
                        .when(pl.col.variable=='fitness')
                        .then(pl.lit('fit'))
                        .otherwise(pl.lit('div'))
        )
        # .filter(pl.col.variable!='fitness')
    ),
    x='idx',
    y='value',
    color='variable',
    facet_row='fitwin',
    line_group='sim_idx'
)
fig.update_yaxes(matches=None)

## older

In [26]:
new_trace.trace_analysis.idea_extractor_cache._tcmp['comparisons'][(-1,0)]

{'matches': [{'startOfFirst': 0, 'startOfSecond': 1, 'length': 3, 'id': 1},
  {'startOfFirst': 3, 'startOfSecond': 33, 'length': 2, 'id': 2}],
 'num_matches': 5,
 'first_idx': -1,
 'second_idx': 0}

In [17]:

import polars as pl

rvs = []

trace_solns = [a for a in new_trace.list if a.is_solution]
trace_solns = list(sorted(trace_solns, key=lambda a: a.fitness, reverse=False))
soln_ranks = {a.trace_id: aidx*1.0/len(trace_solns) for aidx, a in enumerate(trace_solns)}

for p_idx, c_idx in new_trace.graph.edges:
    if new_trace[p_idx].is_solution and new_trace[c_idx].is_solution:
        rvs.append(dict(
            dist=new_trace.trace_analysis.distilled_dist(p_idx, c_idx),
            parent_rank=soln_ranks[p_idx],
            child_rank=soln_ranks[c_idx],
            parent_fitness=new_trace[p_idx].fitness,
            child_fitness=new_trace[c_idx].fitness,
        ))
ddf = pl.DataFrame(rvs)

In [ ]:
import plotly.express as px

px.box(
    (
        ddf
        .with_columns(fitness_delta = pl.col.child_fitness-pl.col.parent_fitness)
        .with_columns(rank_delta = pl.col.child_rank-pl.col.parent_rank)
        .with_columns(abs_rank_delta = pl.col.rank_delta.abs())
        .with_columns(dist_bin = 0.25*(pl.col.dist // 0.25) )
    ),
    x='dist_bin',
    y='abs_rank_delta'
)

In [15]:
ddf

dist,parent_fitness,child_fitness
f64,f64,f64
0.0,-6.460304,-6.463119
0.0,-7.670745,-6.458843
0.548387,-6.329238,-6.322405
0.0,-6.531031,-20.028857
0.954545,-6.364285,-6.623387
…,…,…
0.950588,-5.857978,-5.797379
0.734923,-5.971385,-6.500188
0.464555,-5.967412,-5.797379


In [20]:
def get_idea(x, vintage=max(trace_cache.keys())):
    return trace_cache[vintage].trace_analysis.distilled_ideas[x]

In [ ]:
# why did we get concluded_dissimilar with 38 and 43?

## debugging

In [15]:
from functools import reduce

def print_ideas(idids, new_trace):
    ids = {i: new_trace.trace_analysis.distilled_ideas[i] for i in idids}
    all_solns = reduce(lambda a, b: a.union(b), map(lambda ii: ids[ii].originating_atom.solution_members(), idids), set())
    for ss in all_solns:
        print(f"=====SOLN {ss}======")
        validids = [i for i in idids if ss in ids[i].originating_atom.solution_members()]
        if len(validids) < 2 and len(idids)>1:
            print(f'   (only {validids})')
            continue
        last_id = None
        for ii in validids:
            if last_id is not None:
                print(ids[last_id].snippets[ss])
                print(f"-----{last_id}//{ii}------")
            last_id = ii
        else:
            print(ids[ii].snippets[ss])
        print(f"=================")
        

## viz funcs

In [7]:
import math
import polars as pl

def _ideas_df(trace_res):
    idea_rows = []
    for ii, ir in (trace_res.trace_analysis.idea_extractor_cache._idea_reflections.items()):
        fitnesses = [trace_res[a].fitness for a in ir.solution_members]
        err_rate = len([f for f in fitnesses if not math.isfinite(f)]) / len(fitnesses)
        fin_fitnesses = [f for f in fitnesses if math.isfinite(f)]
        if len(fin_fitnesses) == 0:
            avg_fitness = float('nan')
        else:
            avg_fitness = sum(fin_fitnesses, start=0) / max(len(fin_fitnesses), 1)
        for si in ir.solution_members:
            idea_rows.append(
                dict(
                    solution_id=si,
                    idea_id=ii,
                    err_rate=err_rate,
                    idea_fitness=avg_fitness,
                    idea_membersize=len(ir.solution_members)
                )
            )
    return pl.DataFrame(idea_rows).with_columns(idea_rank=pl.col.idea_fitness.rank(descending=False))

In [8]:
import math
import polars as pl
from IPython.display import display
import plotly.express as px

def _clusterview(trace_res, x_col='log_fitness', y_col='idea_id'):
    _plotdf = (
        pl.DataFrame(
            [(a.trace_id, a.fitness) for a in trace_res.list if a.is_solution],
            schema=[('solution_id', pl.Int32), ('fitness', pl.Float64)]
        )
        .join(
            _ideas_df(trace_res),
            on=['solution_id'],
            how='right'
        )
        .with_columns(pl.col.idea_id.cast(str))
        .with_columns((pl.col.fitness.abs().log() * pl.col.fitness.sign()).alias('log_fitness'))
        .with_columns((pl.col.log_fitness.abs().log() * pl.col.log_fitness.sign()).alias('loglog_fitness'))
        .with_columns(pl.lit(1).alias('count'))
    )

    figa = px.line(
        _plotdf,
        x=x_col,
        # y='idea_id',
        y=y_col,
        color='idea_id',
        hover_data=['solution_id', 'idea_id'],
        markers=True,
        
        title='Idea Clusters across Fitness spectrum'
    )
    
    figb = px.line(
        _plotdf,
        x=x_col,
        # y='idea_id',
        y=y_col,
        line_group='solution_id',
        # markers=True,
        
        title='Nodes across Idea Clusters'
    )
    figb.update_traces(line=dict(color='#212121', width=1.5))

    figc = px.line()  # empty figure
    
    # Add dark gray lines first (background)
    for trace in figb.data:
        figc.add_trace(trace)
    
    # Add colored idea cluster lines on top (foreground)
    for trace in figa.data:
        figc.add_trace(trace)
    
    # Optionally adjust axis/legend settings for clarity
    figc.update_layout(
        title=f'Idea Clusters with Reference Solution Lines, ordered by `{x_col}`', #<br> {"        "*4} <span style="font-size:15px;"> with reference lines for unique solutions
        xaxis_title=x_col,
        yaxis_title=y_col,
    xaxis_type='linear',
    yaxis_type='linear',
        showlegend=False
    )
    return figc


# display( _clusterview(trace_res, x_col='loglog_fitness', y_col='idea_rank') )
# display( _clusterview(trace_res, x_col='solution_id', y_col='idea_rank') )
# display( _clusterview(trace_res, x_col='loglog_fitness', y_col='idea_id') )
# display( _clusterview(trace_res, x_col='solution_id', y_col='idea_id') )


In [74]:
import polars as pl
import plotly.express as px
from itertools import product


def _idea_cooccurrence_df(new_trace, iddf=None):
    if iddf is None:
        iddf = _ideas_df(new_trace)
    uniq_ideas = iddf.select(pl.col.idea_id.unique()).to_series().to_list()
    
    cc = []
    for a,b in product(uniq_ideas,uniq_ideas):
        if a==b: continue
        bb = (
            iddf
            .filter(pl.col.idea_id==a)
            .with_columns(num_solns=pl.col.solution_id.len())
            .join(
                iddf
                .filter(pl.col.idea_id==b)
                .with_columns(num_solns=pl.col.solution_id.len()),
                on=['solution_id']
            )
            .select(
                pl.col.idea_id.alias('a_id'),
                pl.col.idea_id_right.alias('b_id'),
                pl.col.solution_id,
                pl.col.num_solns.alias('a_total'),
                pl.col.num_solns_right.alias('b_total'),
            )
            .group_by(['a_id', 'b_id'])
                .agg(
                    pl.col.solution_id.len().alias('matches'),
                    pl.col.a_total.max(),
                    pl.col.b_total.max(),
                )
            .with_columns(
                min_matchpct = pl.min_horizontal(pl.col.matches/pl.col.a_total, pl.col.matches/pl.col.b_total),
                max_matchpct = pl.max_horizontal(pl.col.matches/pl.col.a_total, pl.col.matches/pl.col.b_total),
                avg_totalsize = pl.mean_horizontal(pl.col.a_total, pl.col.b_total),
            )
        )
        cc.append(bb)
    return pl.concat(cc)

def _soln_cooccurrence_df(new_trace, iddf=None):
    if iddf is None:
        iddf = _ideas_df(new_trace)
    uniq_solns = iddf.select(pl.col.solution_id.unique()).to_series().to_list()
    
    cc = []
    for a,b in product(uniq_solns,uniq_solns):
        if a==b: continue
        bb = (
            iddf
            .filter(pl.col.solution_id==a)
            .with_columns(num_ideas=pl.col.idea_id.len())
            .join(
                iddf
                .filter(pl.col.solution_id==b)
                .with_columns(num_ideas=pl.col.idea_id.len()),
                on=['idea_id']
            )
            .select(
                pl.col.solution_id.alias('a_id'),
                pl.col.solution_id_right.alias('b_id'),
                pl.col.idea_id,
                pl.col.num_ideas.alias('a_total'),
                pl.col.num_ideas_right.alias('b_total'),
            )
            .group_by(['a_id', 'b_id'])
                .agg(
                    pl.col.idea_id.len().alias('matches'),
                    pl.col.a_total.max(),
                    pl.col.b_total.max(),
                )
            .with_columns(
                min_matchpct = pl.min_horizontal(pl.col.matches/pl.col.a_total, pl.col.matches/pl.col.b_total),
                max_matchpct = pl.max_horizontal(pl.col.matches/pl.col.a_total, pl.col.matches/pl.col.b_total),
                avg_totalsize = pl.mean_horizontal(pl.col.a_total, pl.col.b_total),
            )
        )
        cc.append(bb)
    return pl.concat(cc)

def soln_cooccur_view(trace_res):
    cc = _soln_cooccurrence_df(trace_res)
    # Use scatter plot to show nonzero positions
    ranks = {v:i for i, v in enumerate(sorted(cc.select(pl.col.a_id.unique()).to_series().to_list()))}
    dfig = px.scatter(
        cc.with_columns(a_idrank=pl.col.a_id.replace(ranks), b_idrank=pl.col.b_id.replace(ranks))
        .with_columns( pl.struct(["a_id", "b_id"]).map_elements(lambda x: trace_res.trace_analysis.distilled_dist(x['a_id'], x['b_id']), return_dtype=pl.Float64).alias('dist') ),
        x="a_idrank",
        y="b_idrank",
        title="Nonzero Grid Positions",
        color='dist',
        color_continuous_scale=px.colors.sequential.Inferno[::-1],
        size='avg_totalsize',
        hover_data=['a_total', 'b_total', 'avg_totalsize', 'min_matchpct', 'max_matchpct', 'a_id', 'b_id'],
        labels={"col": "Column Index", "row": "Row Index"},
        width=800, height=800,
    )
    
    fig = dfig # empty figure
    
    # Optionally adjust axis/legend settings for clarity
    fig.update_layout(
        title=f'Solution Ideas Co-Occurrences', #<br> {"        "*4} <span style="font-size:15px;"> with reference lines for unique solutions
    )
    
    fig.update_traces(line=dict(color='#212121', width=1.5))
    
    return fig

def idea_cooccur_view(trace_res):
    cc = _idea_cooccurrence_df(trace_res)
    # Use scatter plot to show nonzero positions
    ranks = {v:i for i, v in enumerate(sorted(cc.select(pl.col.a_id.unique()).to_series().to_list()))}
    dfig = px.scatter(
        cc
        .with_columns(a_idrank=pl.col.a_id.replace(ranks), b_idrank=pl.col.b_id.replace(ranks))
        # pl.struct(["a", "b"]).map_batches(
        #     lambda x: test_func(x.struct.field("a"), x.struct.field("b"), 1)
        # )
        ,
        x="a_id",
        y="b_id",
        title="Nonzero Grid Positions",
        color='min_matchpct',
        color_continuous_scale=px.colors.sequential.Inferno[::-1],
        size='avg_totalsize',
        hover_data=['a_total', 'b_total', 'avg_totalsize', 'min_matchpct', 'max_matchpct'],
        labels={"col": "Column Index", "row": "Row Index"},
        width=800, height=800,
    )
    
    fig = dfig # empty figure
    
    # Optionally adjust axis/legend settings for clarity
    fig.update_layout(
        title=f'Idea Solutions Co-Occurrences', #<br> {"        "*4} <span style="font-size:15px;"> with reference lines for unique solutions
    )
    
    fig.update_traces(line=dict(color='#212121', width=1.5))
    
    return fig

## viz

In [48]:

import numpy as np

def distilled_dist( a_idx, b_idx):
    # find ideas overlap
    ab_atoms = [
        ide.originating_atom for ide in new_trace.trace_analysis.distilled_ideas.values()
        if len(ide.originating_atom.solution_members().intersection({a_idx, b_idx}))==2
    ]
    # print(ab_atoms)
    # raise Exception("asdfsd")
    a_tokenlen = new_trace.trace_analysis.idea_extractor_cache._tcmp['submissions'][a_idx]['numberOfTokens']
    b_tokenlen = new_trace.trace_analysis.idea_extractor_cache._tcmp['submissions'][b_idx]['numberOfTokens']
    hasmatch_memo = np.zeros((a_tokenlen,))

    for ide in ab_atoms:
        ide_segm = ide.get(a_idx)
        hasmatch_memo[ide_segm.start_token:ide_segm.end_token+1] = 1
    # print(ab_atoms)
    # print(hasmatch_memo)
    return hasmatch_memo.sum() * (0.5 / a_tokenlen + 0.5 / b_tokenlen)

new_trace.trace_analysis.distilled_dist = distilled_dist

In [49]:
new_trace.trace_analysis.distilled_dist(57,78)

0.9838709677419355

In [70]:

def distilled_dist(self, a_idx, b_idx):
    # find ideas overlap
    ab_atoms = [
        ide.originating_atom for ide in self.distilled_ideas.values()
        if ide.originating_atom.solution_members().intersection({a_idx, b_idx}) == {a_idx, b_idx}
    ]
    a_tokenlen = self.idea_extractor_cache._tcmp['submissions'][a_idx]['numberOfTokens']
    b_tokenlen = self.idea_extractor_cache._tcmp['submissions'][b_idx]['numberOfTokens']
    hasmatch_memo = np.zeros((a_tokenlen,))

    for ide in ab_atoms:
        ide_segm = ide.get(a_idx)
        hasmatch_memo[ide_segm.start_token:ide_segm.end_token+1] = 1
    
    return hasmatch_memo.sum() * (0.5 / a_tokenlen + 0.5 / b_tokenlen)

In [89]:
[1,2,3].pop(0)

1

In [ ]:
%debug

In [14]:
len(new_trace.trace_analysis.distilled_ideas)

105

In [16]:
len(new_trace.trace_analysis.distilled_ideas)

53

# experiments

## re-creation

### REDO

In [162]:

from eallmancy.analysis.trace_analysis.idea_distillation import DistilledIdea

from eallmancy import SinglePartSolution
import random


test_engine = EngineV3(
    problem_spec=TSPACOProblemSpec(maximization=True), api_spec=dict(temperature=1.0, **CHATGPT_4OMINI_API),
    tracing=False
)
test_engine.start()

NEWL_ = """
"""


def _snippet_tag(snip):
    return f"[SNIPPET]{NEWL_}```{snip}```"

def _build_residual_distillation(soln_id, all_snippets, ttrace, engine, tcmp):
    
    pprint = NEWL_.join([_snippet_tag(v) for v in all_snippets])
    prmpt = f"""
A team has been working on implementations of the following function:
```\n{engine._problem_spec.template}```

They have produced many prototypes, some of which have recognizable similarities.
We are very interested in the importance, if any, of the following code snippets found in multiple implementations:
{pprint}

Without referencing any specific solutions by id, specify the core formulas or algorithmic concepts present in the code snippets as a short description of implementation concepts.
You should specify any commonly appearing formulas - assume the reader cannot see the original snippets. 
Your explanation should make clear how implementations containing these code snippets would differ from other implementations, and the role that the code snippets have with regards to the originating prototype. Do not give qualitative feedback.

EXAMPLE1 (your answer can be longer than these as necessary, but with similar content):
```
The cluster's members use SOME_FUNCTION to adjust SOME_PARAMETER in the following form `EQUATION_HERE`.
Variations can include SOME_MODIFICATION, SOME_OTHER_MOD, or SOMETHING_ELSE.
```

EXAMPLE2
```
The cluster's members are implemented as SOME_CODE_STRUCTURE, where SOME_CHARACTERISTIC is frequently present.
Examples for SOME_DETAIL include SOME_METHOD_A, SOME_METHOD_B, or SOME_METHOD_C.
```

Make sure to keep your focus on the content of the code snippets, only using the overall solutions only for contextual details.
Do not give markdown headers, just provide your analysis in several or fewer short paragraphs along with equations or short code examples.
"""
    art, _ = engine.build_artifact(prmpt, DistilledIdea, register=False)
    art.solution_members = {soln_id}
    art.originating_atom = None
    art.snippets = all_snippets

    # TODO: remove registration! really should be handled elsewhere...
    engine.register_artifact(art, parents=[ttrace[p] for p in art.solution_members], disable_hooks=True, operation_name="IdeaDistillation")

    return art


# def build_residual_distillation(soln_id, trace_res):
#     return _build_residual_distillation(
#         soln_id,
#         trace_res,
#         trace_res.trace_analysis.idea_extractor_cache._internal_engine,
#         trace_res.trace_analysis.idea_extractor_cache._tcmp,
#     )

    


In [163]:
from eallmancy.analysis.trace_analysis.segm_printing import simple_segm_substr, _find_token_line_indices, _find_line_stidx, _find_line_enidx

import numpy as np


def _unmatched_code_lines(soln_id, ide, trace_res):
    token_list = trace_res.trace_analysis.idea_extractor_cache._tcmp['submissions'][soln_id]['tokenList']
    soln_num_tokens = len(token_list)
    hasmatch_memo = np.zeros((soln_num_tokens,))
    
    ide_segm = ide.originating_atom.get(soln_id)
    hasmatch_memo[ide_segm.start_token:ide_segm.end_token+1] += 1
    
    unmatched_segments = []
    segm_start = 0
    inseq = False
    for i in range(soln_num_tokens):
        if hasmatch_memo[i] != 0 and inseq:
            unmatched_segments.append((segm_start, i))
            inseq = False
        elif  hasmatch_memo[i] == 0 and not inseq:
            segm_start = i
            inseq = True
    else:
        if inseq:
            unmatched_segments.append((segm_start, i+1))
    all_strs = [simple_segm_substr(trace_res[soln_id].program, a, b-1, token_list) for a,b in unmatched_segments]
    return all_strs


In [164]:


def _build_idea_residuals(ide, trace_res, engine):
    ress = dict()
    atm = ide.originating_atom
    for s in atm.solution_members():
        # grab unmatched segments
        unmatched_snippets = _unmatched_code_lines(s, ide, trace_res)
        # build residual art
        s_res = _build_residual_distillation(s, unmatched_snippets, trace_res, engine, trace_res.trace_analysis.idea_extractor_cache._tcmp)
        ress[s] = s_res
    return ress


In [165]:


def recreate_solution(soln_id, original_trace, engine, relevant_ideas=None):
    if relevant_ideas is None:
        raise Exception('must give at least one relevant idea to `recreate_solution`')
    
    random.shuffle(relevant_ideas)
    
    pmpt = f"""
You are an expert in the domain of optimization heuristics. Your task is to design heuristics that can effectively solve optimization problems.

Below are high-level documentations of concepts in a `{engine._problem_spec.entrypoint}` implementation for {engine._problem_spec.description}
{engine._problem_spec.func_description}

[Example]
{engine._problem_spec.template}

[Implementation Descriptions]
{f'{NEWL_*2}'.join([f'Description {idx}{NEWL_}{ri.analysis}' for idx, ri in enumerate(relevant_ideas)])}


[Code]
Please recreate the implementation of `{engine._problem_spec.entrypoint}`, using the documented elements. Output code only and enclose your code with Python code block: ```python ... ```.
"""
    # return pmpt
    return engine.generate_artifact(pmpt, SinglePartSolution)


In [166]:
from enum import Enum
from collections import defaultdict

class RecreateMode(Enum):
    FULL = 1
    MATCH = 2
    RES = 3

In [101]:
print(aaa[56].analysis)

The implementation uses the concept of heuristic distance calculations with the formula `1 / (distance_matrix + damping_factor)`, where the damping factor is derived from multiplying the identity matrix by a large scalar to avoid division by zero or amplify negligible distances. This initial transformation creates a heuristic matrix that inversely relates to distances, giving higher values to shorter distances.

Normalization of this heuristic matrix is performed using the equation `normalized_matrix = heuristic_matrix / max_value` to ensure that all values are within the range [0, 1]. The presence of `max_value > 0` guards against division by zero when normalizing.

The method also introduces a dynamic threshold for edge selection based on a percentile rank using `threshold = np.percentile(normalized_matrix[normalized_matrix > 0], 75)`. This allows for the retention of the top 25% of relevant edges from the normalized matrix. In this context, edges with values below the threshold are 

### residual distillation logic

In [23]:
from eallmancy.analysis.trace_analysis.segm_printing import simple_segm_substr, _find_token_line_indices, _find_line_stidx, _find_line_enidx


import numpy as np


def _get_matching_ideas(soln_id, trace_res):
    id_ls = []
    for ide in trace_res.trace_analysis.idea_extractor_cache._idea_reflections.values():
        if soln_id in ide.solution_members:
            id_ls.append(ide)
    return id_ls

def _unmatched_code_lines(soln_id, trace_res):
    token_list = trace_res.trace_analysis.idea_extractor_cache._tcmp['submissions'][soln_id]['tokenList']
    soln_num_tokens = len(token_list)
    hasmatch_memo = np.zeros((soln_num_tokens,))
    
    for ide in _get_matching_ideas(soln_id, trace_res):
        ide_segm = ide.originating_atom.get(soln_id)
        hasmatch_memo[ide_segm.start_token:ide_segm.end_token+1] += 1
    
    unmatched_segments = []
    segm_start = 0
    inseq = False
    for i in range(soln_num_tokens):
        if hasmatch_memo[i] != 0 and inseq:
            unmatched_segments.append((segm_start, i))
            inseq = False
        elif  hasmatch_memo[i] == 0 and not inseq:
            segm_start = i
            inseq = True
    else:
        if inseq:
            unmatched_segments.append((segm_start, i+1))
    all_strs = [simple_segm_substr(trace_res[soln_id].program, a, b-1, token_list) for a,b in unmatched_segments]
    return all_strs

# uml = (_unmatched_code_lines(59, new_trace))
# print(uml)

### re-creation operation

In [34]:

from eallmancy import SinglePartSolution
import random


test_engine = EngineV3(
    problem_spec=TSPACOProblemSpec(maximization=True), api_spec=dict(temperature=1.0, **CHATGPT_4OMINI_API),
    tracing=False
)
test_engine.start()

def fetch_relevant_ideas(soln_id, trace, residual_distillation=True):
    relevant_ideas = [idd for idd in trace.trace_analysis.distilled_ideas.values() if soln_id in idd.solution_members]

    if residual_distillation:
        relevant_ideas.append( build_residual_distillation(soln_id, trace) )
    return relevant_ideas

def recreate_solution(soln_id, original_trace, engine, relevant_ideas=None, residual_distillation=True):
    if relevant_ideas is None:
        relevant_ideas = fetch_relevant_ideas(soln_id, original_trace, residual_distillation)
    if len(relevant_ideas) == 0:
        return None
    
    random.shuffle(relevant_ideas)
    
    pmpt = f"""
You are an expert in the domain of optimization heuristics. Your task is to design heuristics that can effectively solve optimization problems.

Below are high-level documentations of concepts in a `{engine._problem_spec.entrypoint}` implementation for {engine._problem_spec.description}
{engine._problem_spec.func_description}

[Example]
{engine._problem_spec.template}

[Implementation Descriptions]
{f'{NEWL_*2}'.join([f'Description {idx}{NEWL_}{ri.analysis}' for idx, ri in enumerate(relevant_ideas)])}


[Code]
Please recreate the implementation of `{engine._problem_spec.entrypoint}`, using the documented elements. Output code only and enclose your code with Python code block: ```python ... ```.
"""
    # return pmpt
    return engine.generate_artifact(pmpt, SinglePartSolution)


In [29]:
%%time

res_distillations = dict()
for ss in list(new_trace.graph.nodes):
    if ss in res_distillations.keys():
        continue
    if new_trace[ss].is_solution:
        res_distillations[ss] = build_residual_distillation(ss, new_trace)


CPU times: user 826 ms, sys: 50.6 ms, total: 877 ms
Wall time: 4min 8s


In [30]:
recr_results = dict()
recr_results[RecreateMode.FULL] = defaultdict(lambda : [])
recr_results[RecreateMode.MATCH] = defaultdict(lambda : [])
recr_results[RecreateMode.RES] = defaultdict(lambda : [])

In [ ]:
%%prun

n_recrs=3

for iii in range(n_recrs):
    print(f'RUN {iii}')
    for trace_id in list(new_trace.graph.nodes):
        if not new_trace[trace_id].is_solution:
            continue
        matched_ideas = fetch_relevant_ideas(trace_id, new_trace, residual_distillation=False)
        recr_results[RecreateMode.FULL][trace_id].append( recreate_solution(trace_id, new_trace, test_engine, relevant_ideas=matched_ideas+[res_distillations[trace_id]]) )
    print(f'RUN {iii}: FULL complete')
        
    for trace_id in list(new_trace.graph.nodes):
        if not new_trace[trace_id].is_solution:
            continue
        matched_ideas = fetch_relevant_ideas(trace_id, new_trace, residual_distillation=False)
        recr_results[RecreateMode.MATCH][trace_id].append( recreate_solution(trace_id, new_trace, test_engine, relevant_ideas=matched_ideas) )
    print(f'RUN {iii}: MATCH complete')



    for trace_id in list(new_trace.graph.nodes):
        if not new_trace[trace_id].is_solution:
            continue
        matched_ideas = fetch_relevant_ideas(trace_id, new_trace, residual_distillation=False)
        recr_results[RecreateMode.RES][trace_id].append( recreate_solution(trace_id, new_trace, test_engine, relevant_ideas=[res_distillations[trace_id]]) )
    print(f'RUN {iii}: RES complete')
    

In [111]:
xxx = fetch_relevant_ideas(39, trace_res, residual_distillation=False)
xxx

In [122]:
print(xxx[0].snippets[4])

    heuristic_matrix = np.zeros_like(distance_matrix)
    
    # Calculate the sum of distances from each node to all other nodes
    total_distances = np.sum(distance_matrix, axis=1)


In [121]:
print(xxx[0].snippets[39])

    heuristic_matrix = np.zeros_like(distance_matrix)

    # Calculate the average distance from each node to all other nodes
    avg_distance = np.mean(distance_matrix, axis=1)


In [70]:
print(trace_res[25])


Artifact 25
```
import numpy as np

def heuristic(distance_matrix: np.ndarray) -> np.ndarray:
    # Inverse of distances to prioritize shorter paths
    inverse_distances = 1 / (distance_matrix + np.finfo(float).eps)  # Prevent division by zero
    
    # Calculate the average distance excluding zeros or extreme values
    valid_distances = distance_matrix[distance_matrix > 0]
    adaptive_threshold = np.percentile(valid_distances, 75)  # Use the 75th percentile as an adaptive threshold
    
    # Calculate promising factors based on the average of valid distances
    average_distance = np.mean(valid_distances)
    promising_factors = np.clip(average_distance / (distance_matrix + np.finfo(float).eps), 0, None)
    
    # Blend the metrics to emphasize promising edges
    score_matrix = inverse_distances * promising_factors
    
    # Sparsify the score matrix based on the adaptive threshold
    score_matrix[distance_matrix >= adaptive_threshold] = 0
    
    return score_matrix

```
E

In [43]:

# def build_residual_distillation(soln_id, ttrace, engine, tcmp):

xx = build_residual_distillation(59, trace_res)

xx

In [44]:
print(xx.analysis)

The implementation utilizes a mechanism to prevent division by zero by incorporating a small constant, represented by `eps`, in the formula for calculating inverse distances. This is achieved through the expression `1 / (distance_matrix + eps)`, ensuring that distances that are zero do not lead to undefined behavior. The valid mask, created using `valid_mask = distance_matrix > 0`, filters the distance matrix, allowing the algorithm to compute inverse distances only for non-zero elements.

Furthermore, the implementation introduces an adaptive threshold based on the 75th percentile of valid distances. This threshold is computed using the formula `adaptive_threshold = np.percentile(valid_distances, 75)`, where `valid_distances` are those distances greater than zero. This adaptive threshold plays a critical role in determining the promising factors, calculated as `promising_factors = np.clip(adaptive_threshold / (distance_matrix + eps), 0, None)`. This approach allows the algorithm to dy

In [17]:
print(trace_res[0])


Artifact 0
```
import numpy as np

def heuristic(distance_matrix: np.ndarray) -> np.ndarray:
    # Avoid division by zero and set up a mask
    mask = distance_matrix > 0
    promising_edges = np.zeros_like(distance_matrix)

    # Calculate heuristic based on distance, inverse and normalization
    inverse_distances = np.where(mask, 1 / distance_matrix, 0)
    
    # Calculate row sums for normalization
    row_sums = np.sum(inverse_distances, axis=1, keepdims=True)
    
    # Normalize to get a score between 0 and 1 for promising edges
    with np.errstate(divide='ignore', invalid='ignore'):
        normalized_scores = np.divide(inverse_distances, row_sums, where=row_sums != 0)
    
    # Apply a sparsity filter to keep only edges that have high promising scores (above a threshold)
    threshold = np.percentile(normalized_scores[mask], 75)  # Keep the top 25%
    promising_edges[normalized_scores >= threshold] = normalized_scores[normalized_scores >= threshold]
    
    return promis

In [18]:
print(trace_res[40])


Artifact 40
```
import numpy as np

def heuristic(distance_matrix: np.ndarray) -> np.ndarray:
    # Prevent division by zero by adding a small constant
    eps = np.finfo(float).eps
    inverse_distances = 1 / (distance_matrix + eps)

    # Calculate valid distances and set adaptive threshold using the 75th percentile
    valid_distances = distance_matrix[distance_matrix > 0]
    adaptive_threshold = np.percentile(valid_distances, 75) if valid_distances.size > 0 else np.inf

    # Calculate promising factors using a distance-dependent score
    promising_factors = np.clip(adaptive_threshold / (distance_matrix + eps), 0, None)

    # Compute the score matrix combining inverse distances and promising factors
    score_matrix = inverse_distances * promising_factors
    
    # Zero out scores for distances above the adaptive threshold
    score_matrix[distance_matrix >= adaptive_threshold] = 0
    
    # Normalize score matrix to ensure values are between 0 and 1
    max_score = np.max(sc

In [22]:
print(trace_res[54])

Incorporate adaptive thresholds, avoid division by zero, and normalize scores for improved evaluation of promising edges.


In [21]:
print(uml[0])

import numpy as np

def heuristic(distance_matrix: np.ndarray) -> np.ndarray:
    # Prevent division by zero by adding a small constant
    eps = np.finfo(float).eps
    valid_mask = distance_matrix > 0
    inverse_distances = np.where(valid_mask, 1 / (distance_matrix + eps), 0)

    # Calculate valid distances and set adaptive threshold using the 75th percentile



In [16]:
trace_res.graph.in_edges(59)

InEdgeDataView([(0, 59), (40, 59), (54, 59)])